# RAG-Based Intelligent Document Q&A System

This project builds a Retrieval-Augmented Generation system for long-document question answering.  
The goal is to reduce hallucination by retrieving relevant document chunks before generating answers with an LLM.

Large language models can generate fluent but unsupported answers when asked about private or long documents.  
This project uses document chunking, embeddings, vector search, reranking, and evidence-grounded generation to improve answer reliability.

## 1. Load Documents

In [8]:
!pip install pymupdf sentence-transformers faiss-cpu pandas numpy tqdm

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 41.5/41.5 kB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB 262.6 kB/s eta 0:01:14
   ---------------------------------------- 0.0/19.2 MB 281.8 kB/s eta 0:01:09
   ---------------------------------------- 0.1/19.2 MB 363.1 kB/s eta 0:00:53
   ---------------------------------------- 0.1/19.2 MB 467.6 kB/s eta 0:00:41
   ---------------------------------------- 0.1/19.2 MB 467.6 kB/s eta 0:00:41
   ---------------------------------------- 0.1/19.2 MB 425.3 kB/s eta 0:00:45
   ---------------------------------------- 0.2/19.2 MB 476.3 kB/s eta 0:00:41
   ---------------------------------------- 0.2/19.2 MB 529.7 kB/s eta 0:00:36
    --------------------------------------- 0.3/19.2 MB 561.1 kB/s eta 0:00:

In [12]:
import fitz
import pandas as pd
import numpy as np
from tqdm import tqdm

pdf_path = "Artificial_ Intelligence_Index_Report_2025.pdf"

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_idx, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page": page_idx + 1,
            "text": text
        })

    return pages

pages = extract_pdf_text(pdf_path)

print("Number of pages:", len(pages))
print(pages[0]["text"][:1000])

Number of pages: 457
Artificial Intelligence
Index Report 2025



## 2. Check the amount of text on each page

In [19]:
page_df = pd.DataFrame(pages)
page_df["text_length"] = page_df["text"].apply(len)

page_df[["page", "text_length"]].head(20)

,page,text_length
0,1,42
1,2,2191
2,3,2466
3,4,3000
4,5,3026
5,6,758
6,7,1365
7,8,1884
8,9,96
9,10,2879


In [21]:
page_df["text_length"].describe()

count     457.000000
mean     1797.470460
std       932.849212
min        42.000000
25%      1092.000000
50%      1663.000000
75%      2316.000000
max      4690.000000
Name: text_length, dtype: float64

## 3. Text Chunking

In [29]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [31]:
all_chunks = []

for page in pages:
    chunks = chunk_text(page["text"], chunk_size=800, overlap=150)

    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "source": "Stanford AI Index Report 2025",
            "page": page["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
chunks_df.head()

Total chunks: 1421


,source,page,chunk_id,text
0,Stanford AI Index Report 2025,2,0,Artificial Intelligence\nIndex Report 2025\n1\...
1,Stanford AI Index Report 2025,2,1,n offshoot of the One Hundred Year Study of Ar...
2,Stanford AI Index Report 2025,2,2,the rapid evolution of underlying technologies...
3,Stanford AI Index Report 2025,2,3,the world. We have briefed companies like Acce...
4,Stanford AI Index Report 2025,3,0,Artificial Intelligence\nIndex Report 2025\n2\...


## 4. Build Embeddings

In [36]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=False,
    convert_to_numpy=True
)

embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(1421, 384)

## 5. Build Vector Search with FAISS

In [38]:
import faiss

embeddings = embeddings.astype("float32")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 1421


## 6. Retrieval search

In [43]:
def retrieve(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]

        results.append({
            "rank": rank + 1,
            "page": row["page"],
            "chunk_id": row["chunk_id"],
            "distance": float(distances[0][rank]),
            "text": row["text"]
        })

    return results

Test a question

In [46]:
query = "What are the major trends in AI model performance and deployment?"

results = retrieve(query, top_k=5)

for r in results:
    print(f"Rank: {r['rank']}")
    print(f"Page: {r['page']}")
    print(f"Distance: {r['distance']:.4f}")
    print(r["text"][:600])
    print("-" * 100)

Rank: 1
Page: 100
Distance: 0.7009
ins, combined with growing datasets, could lead to 
even higher-performing models. Additionally, inference on 
smaller models is typically faster and less expensive. Their 
emergence also lowers the barrier to entry for AI developers 
and businesses looking to integrate AI into their operations.
2.1 Overview of AI in 2024
Chapter 2: Technical Performance
PaLM
LLaMA-65B
Llama 2 34B
Mistral 7B
Phi-3-mini
2022-May
2022-Sep
2023-Jan
2023-May
2023-Sep
2024-Jan
2024-May
10B
100B
Publication date
Number of parameters (log scale)
Smallest AI models scoring above 60% on MMLU, 2022–24
Source: Abdin et a
----------------------------------------------------------------------------------------------------
Rank: 2
Page: 13
Distance: 0.7380
es continues to be the leading source of notable AI models. In 2024, U.S.-based institutions 
produced 40 notable AI models, significantly surpassing China’s 15 and Europe’s combined total of three. In the past decade, 
more nota

## 7. RAG answer generation

In [48]:
def show_results(query, top_k=5):
    results = retrieve(query, top_k=top_k)

    display_df = pd.DataFrame([
        {
            "rank": r["rank"],
            "page": r["page"],
            "distance": r["distance"],
            "preview": r["text"][:300].replace("\n", " ")
        }
        for r in results
    ])

    return display_df

show_results("How has AI investment changed in recent years?", top_k=5)

,rank,page,distance,preview
0,1,359,0.419033,358 Artificial Intelligence Index Report 2025 ...
1,2,248,0.433911,s investment section in the AI Index includes...
2,3,249,0.454453,248 Artificial Intelligence Index Report 2025 ...
3,4,219,0.514232,218 Artificial Intelligence Index Report 2025 ...
4,5,361,0.550424,360 Artificial Intelligence Index Report 2025 ...


## 8. Evaluation

In [50]:
test_queries = [
    "What are the main trends in AI investment?",
    "How is AI being used in healthcare?",
    "What does the report say about foundation models?",
    "What are the trends in AI publications and patents?",
    "How has the cost of AI model inference changed?",
    "What are the risks or challenges related to AI adoption?",
    "What does the report say about AI education?"
]

for q in test_queries:
    print("Question:", q)
    display(show_results(q, top_k=3))
    print("=" * 120)

Question: What are the main trends in AI investment?


,rank,page,distance,preview
0,1,359,0.503436,358 Artificial Intelligence Index Report 2025 ...
1,2,249,0.511238,248 Artificial Intelligence Index Report 2025 ...
2,3,248,0.534340,s investment section in the AI Index includes...


Question: How is AI being used in healthcare?


,rank,page,distance,preview
0,1,284,0.651573,283 Artificial Intelligence Index Report 2025 ...
1,2,316,0.656951,315 Artificial Intelligence Index Report 2025 ...
2,3,167,0.691648,"algorithms that avoid bias or discrimination, ..."


Question: What does the report say about foundation models?


...

,rank,page,distance,preview
0,1,65,0.636856,Table of Contents 64 Artificial Intelligence I...
1,2,29,0.667637,Table of Contents 28 Artificial Intelligence I...
2,3,100,0.706136,"ins, combined with growing datasets, could lea..."


Question: What are the risks or challenges related to AI adoption?


,rank,page,distance,preview
0,1,183,0.631093,182 Artificial Intelligence Index Report 2025 ...
1,2,183,0.702352,"iversity and nondiscrimination risks (e.g., fa..."
2,3,3,0.736097,"anwhile, AI adoption has accelerated at an un..."


Question: What does the report say about AI education?


,rank,page,distance,preview
0,1,370,0.535590,369 Artificial Intelligence Index Report 2025 ...
1,2,370,0.557934,"mitigating data biases, etc.). For the purpose..."
2,3,378,0.600740,377 Artificial Intelligence Index Report 2025 ...


Test

In [55]:
query = "How has AI investment changed in recent years?"

results = retrieve(query, top_k=3)

for r in results:
    print("=" * 100)
    print(f"Rank: {r['rank']}")
    print(f"Page: {r['page']}")
    print(f"Chunk ID: {r['chunk_id']}")
    print(f"Distance: {r['distance']:.4f}")
    print("-" * 100)
    print(r["text"])

Rank: 1
Page: 359
Chunk ID: 0
Distance: 0.4190
----------------------------------------------------------------------------------------------------
358
Artificial Intelligence
Index Report 2025
Table of Contents
Chapter 6 Preview
6.3 Public Investment in AI
Chapter 6: Policy and Governance
Figure 6.3.6 illustrates the trends in public AI investment 
over time across two significant regions of AI investment, the 
United States and Europe. Both regions have seen substantial 
growth in AI-related spending over the past decade. Notably, 
Europe’s total AI investment in 2023 was approximately 67 
times higher than in 2013, compared to a fifteenfold increase 
in the United States. Europe experienced particularly sharp 
increases in investment, with a 400% year-over-year increase 
in 2017, followed by another major spike of 200% year-over-
year in 2019—a year that also saw a peak in the number of 
national AI strategies released globally. Th
Rank: 2
Page: 248
Chunk ID: 1
Distance: 0.4339
----